**Overview:**

In this project, I will be running 3 large generative LLMs and a much smaller fine-tuned specialized model to examine and classify safe and phishing emails from Kaggle datasets.

**Goal:**

The goal of this experiment to examine which type of model is best fit in a real world application for a system that filters emails and warns its users of their safety.

**Research Question:**

How do instruction-tuned LLMs compare to a fine-tuned small transformer for phishing email classification, and which approach is most effective under identical evaluation conditions?

Models Used:
  1. Mistral 7B Instruct: https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
  2. LLaMA 3.1 8B Instruct: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
  3. Qwen 2.5 7B Instruct: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
  4. RoBERTa-base 100M: https://huggingface.co/FacebookAI/roberta-base

Datasets Used For Testing with LLMs:
  1. Set1: https://www.kaggle.com/datasets/saherpervaiz/phishing-emails-dataset?resource=download
  2. Set2: https://www.kaggle.com/datasets/subhajournal/phishingemails

Datasets Used For Fine-Tuning:
  1. https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset
  2. https://www.kaggle.com/datasets/akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000

In [ ]:
!pip install transformers accelerate bitsandbytes #dependencies
!pip install transformers torch pandas scikit-learn datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Load Clean Dataset & Paths

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from huggingface_hub import login
from google.colab import userdata

login('hugging-face-token')

df = pd.read_csv('path-to-file')
print(f"Full dataset: {df.shape}")
print(df['label'].value_counts())

RESULTS_PATH = 'path-to-folder'

Prompt for LLMs

In [ ]:
def build_prompt(email_text):
    return f"""You are a cybersecurity expert. Classify the following email as either "phishing" or "safe".
Respond with one word only: phishing or safe.

Email:
{email_text[:2000]}

Classification:"""

Inference function

In [ ]:
def run_inference(model, tokenizer, df, model_name, batch_size=32, checkpoint_every=500):
    results = []
    checkpoint_path = f"{RESULTS_PATH}{model_name}_checkpoint.csv"

    # Resume from checkpoint if exists
    try:
        checkpoint_df = pd.read_csv(checkpoint_path)
        results = checkpoint_df.to_dict('records')
        start_idx = len(results)
        print(f"Resuming from index {start_idx}")
    except FileNotFoundError:
        start_idx = 0
        print("Starting fresh")

    for i in range(start_idx, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]

        for _, row in batch.iterrows():
            prompt = build_prompt(row['email_text'])
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

            with torch.no_grad():
                output = model.generate(**inputs, max_new_tokens=5, temperature=0.1)

            decoded = tokenizer.decode(output[0], skip_special_tokens=True)
            response = decoded.split("Classification:")[-1].strip().lower()

            # Parse response to 0 or 1
            if "phishing" in response:
                pred = 1
            else:
                pred = 0

            results.append({
                'email_text': row['email_text'],
                'true_label': row['label'],
                'predicted_label': pred,
                'raw_response': response
            })

        # Checkpoint every N emails
        if (i + batch_size) % checkpoint_every == 0:
            pd.DataFrame(results).to_csv(checkpoint_path, index=False)
            print(f"Checkpoint saved at index {i + batch_size} / {len(df)}")

    # Save final results
    final_path = f"{RESULTS_PATH}{model_name}_results.csv"
    pd.DataFrame(results).to_csv(final_path, index=False)
    print(f"Done! Results saved to {final_path}")
    return pd.DataFrame(results)

Mistral

In [ ]:
from transformers import BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading Mistral...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config
)
print("Mistral loaded. Starting inference...")

mistral_results = run_inference(model, tokenizer, df, "mistral", batch_size=32)

# Quick check
print(mistral_results['predicted_label'].value_counts())
print(mistral_results[['true_label', 'predicted_label', 'raw_response']].head(10))

# Clear VRAM
del model, tokenizer
torch.cuda.empty_cache()
print("Mistral cleared from memory")

Qwen

In [ ]:
from transformers import BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading Qwen...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config
)
print("Qwen loaded. Starting inference...")

qwen_results = run_inference(model, tokenizer, df, "qwen", batch_size=32)

# Quick check
print(qwen_results['predicted_label'].value_counts())
print(qwen_results[['true_label', 'predicted_label', 'raw_response']].head(10))

del model, tokenizer
torch.cuda.empty_cache()
print("Qwen cleared from memory")

Fixed LLaMA Prompt

In [ ]:
from transformers import BitsAndBytesConfig
import pandas as pd
import torch

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading LLaMA...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config
)
print("LLaMA loaded. Starting inference...")

def run_inference_llama(model, tokenizer, df, model_name, batch_size=32, checkpoint_every=500):
    results = []
    checkpoint_path = f"{RESULTS_PATH}{model_name}_checkpoint.csv"

    try:
        checkpoint_df = pd.read_csv(checkpoint_path)
        results = checkpoint_df.to_dict('records')
        start_idx = len(results)
        print(f"Resuming from index {start_idx}")
    except FileNotFoundError:
        start_idx = 0
        print("Starting fresh")

    for i in range(start_idx, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]

        for _, row in batch.iterrows():
            messages = [
                {
                    "role": "system",
                    "content": "You are a cybersecurity expert. Classify emails as phishing or safe. Respond with one word only: phishing or safe."
                },
                {
                    "role": "user",
                    "content": f"Classify this email:\n\n{row['email_text'][:2000]}"
                }
            ]

            # Extract tensor explicitly
            encoded = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt"
            )

            if hasattr(encoded, 'input_ids'):
                input_ids = encoded.input_ids.to("cuda")
            else:
                input_ids = encoded.to("cuda")

            # Create attention mask
            attention_mask = torch.ones_like(input_ids).to("cuda")
            input_length = input_ids.shape[1]

            with torch.no_grad():
                output = model.generate(
                    input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=10,
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )

            new_tokens = output[0][input_length:]
            response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

            if 'phishing' in response:
                pred = 1
            elif 'safe' in response:
                pred = 0
            else:
                pred = -1

            results.append({
                'email_text': row['email_text'],
                'true_label': row['label'],
                'predicted_label': pred,
                'raw_response': response
            })

        if (i + batch_size) % checkpoint_every == 0:
            pd.DataFrame(results).to_csv(checkpoint_path, index=False)
            print(f"Checkpoint saved at index {i + batch_size} / {len(df)}")

    final_path = f"{RESULTS_PATH}{model_name}_results.csv"
    pd.DataFrame(results).to_csv(final_path, index=False)
    print(f"Done! Results saved to {final_path}")
    return pd.DataFrame(results)

llama_results = run_inference_llama(model, tokenizer, df, "llama", batch_size=32)

print(llama_results['predicted_label'].value_counts())
print(llama_results[['true_label', 'predicted_label', 'raw_response']].head(10))

del model, tokenizer
torch.cuda.empty_cache()
print("LLaMA cleared from memory")

Check data across all models

In [ ]:
import pandas as pd
#load results from drive

mistral_results = pd.read_csv(f"{RESULTS_PATH}mistral_results.csv")
llama_results   = pd.read_csv(f"{RESULTS_PATH}llama_results.csv")
qwen_results    = pd.read_csv(f"{RESULTS_PATH}qwen_results.csv")

for name, results in [("Mistral", mistral_results), ("LLaMA", llama_results), ("Qwen", qwen_results)]:
    total = len(results)
    phishing = results['predicted_label'].sum()
    safe = total - phishing
    print(f"{name}: {total} emails → {phishing} phishing, {safe} safe")

________________________________________________________________________________

RoBERTa-base (Not Fine-Tuned)

In [ ]:
MODEL_NAME = "FacebookAI/roberta-base"

print("Loading RoBERTa base...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model = model.to("cuda")
model.eval()
print("RoBERTa loaded.")

In [ ]:
def run_roberta_inference(model, tokenizer, df, model_name, batch_size=64):
    results = []

    print(f"Starting inference on {len(df)} emails...")

    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]

        inputs = tokenizer(
            batch['email_text'].tolist(),
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        for j, (_, row) in enumerate(batch.iterrows()):
            results.append({
                'email_text': row['email_text'],
                'true_label': row['label'],
                'predicted_label': int(predictions[j])
            })

        if (i + batch_size) % 2000 == 0:
            print(f"Progress: {i + batch_size} / {len(df)}")

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{RESULTS_PATH}{model_name}_results.csv", index=False)
    print(f"Done! Saved to {model_name}_results.csv")
    return results_df

roberta_baseline_results = run_roberta_inference(
    model, tokenizer, df, "roberta_baseline", batch_size=64
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("\nRoBERTa Baseline Results:")
print(f"Total emails: {len(roberta_baseline_results)}")
print(f"\nPrediction distribution:")
print(roberta_baseline_results['predicted_label'].value_counts())

print(f"\nAccuracy: {accuracy_score(roberta_baseline_results['true_label'], roberta_baseline_results['predicted_label']):.4f}")
print(f"F1 Score: {f1_score(roberta_baseline_results['true_label'], roberta_baseline_results['predicted_label']):.4f}")
print(f"\nDetailed Report:")
print(classification_report(
    roberta_baseline_results['true_label'],
    roberta_baseline_results['predicted_label'],
    target_names=['Safe', 'Phishing']
))

# Clear memory
del model
torch.cuda.empty_cache()
print("RoBERTa cleared from memory")

RoBERTa-base (Fine-Tuned)

In [ ]:
import torch
import pandas as pd
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report

MODEL_NAME = "eduardocastellon/roberta-phishing-email-detector"


# Load fine-tuned model from HuggingFace
print("Loading fine-tuned RoBERTa from HuggingFace...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to("cuda")
model.eval()
print("Model loaded.")

roberta_finetuned_results = run_roberta_inference(
    model, tokenizer, df, "roberta_finetuned", batch_size=64
)

LLaMA (Original) [DO NOT RUN] (Kept for documentation)

In [ ]:
# from transformers import BitsAndBytesConfig

# MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

# print("Loading LLaMA...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     dtype=torch.float16,
#     device_map="auto",
#     quantization_config=bnb_config
# )
# print("LLaMA loaded. Starting inference...")

# llama_results = run_inference(model, tokenizer, df, "llama", batch_size=32)

# # Quick check
# print(llama_results['predicted_label'].value_calls())
# print(llama_results[['true_label', 'predicted_label', 'raw_response']].head(10))

# del model, tokenizer
# torch.cuda.empty_cache()
# print("LLaMA cleared from memory")